In [1]:
import numpy as np
import rasterio

# --- Input rasters (update paths if needed) ---
r100 = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat.tif"
r50  = r"D:\Phd Research\Final_Raster\50-yr_compound_flood_stat.tif"
r20  = r"D:\Phd Research\Final_Raster\20yr_compound_flood_stat.tif"

# Consider anything > 0 as inundated (tweak if you prefer a noise floor, e.g., 0.01)
INUNDATION_THRESHOLD = 0.0
DEPTH_THRESHOLD_M = 1.0  # the 1 m threshold

def read_raster(path):
    with rasterio.open(path) as ds:
        a = ds.read(1).astype("float64")
        nodata = ds.nodata
        valid = np.isfinite(a)
        if nodata is not None:
            valid &= (a != nodata)
        # pixel area in map units (usually m^2)
        px_area = abs(ds.transform.a * ds.transform.e)
    return a, valid, px_area

def summarize(path):
    arr, valid, px_area = read_raster(path)
    # valid domain cells
    domain_cells = valid.sum()
    # inundated cells (> INUNDATION_THRESHOLD)
    inundated = valid & (arr > INUNDATION_THRESHOLD)
    inund_cells = inundated.sum()
    # cells exceeding threshold (>= 1 m)
    over1m = valid & (arr >= DEPTH_THRESHOLD_M)
    over1_cells = over1m.sum()
    return {
        "domain_cells": domain_cells,
        "inund_cells": inund_cells,
        "over1_cells": over1_cells,
        "domain_area_m2": domain_cells * px_area,
        "inund_area_m2": inund_cells * px_area,
        "over1_area_m2": over1_cells * px_area,
    }

s100 = summarize(r100)
s50  = summarize(r50)
s20  = summarize(r20)

# X: Among inundated 100-yr cells, % that exceed 1 m
X = 100.0 * s100["over1_cells"] / s100["inund_cells"] if s100["inund_cells"] > 0 else np.nan

# Y: Of the whole domain, % of cells that exceed 1 m in the 100-yr case
Y = 100.0 * s100["over1_cells"] / s100["domain_cells"]

# Also compute "larger than" percentages for >1 m area (cells and area give same %)
def pct_larger(a, b):
    return 100.0 * (a / b - 1.0) if b > 0 else np.nan

larger_vs_50 = pct_larger(s100["over1_cells"], s50["over1_cells"])
larger_vs_20 = pct_larger(s100["over1_cells"], s20["over1_cells"])

print(f"X (100-yr: % of inundated cells ≥1 m): {X:.2f}%")
print(f"Y (100-yr: % of entire domain ≥1 m): {Y:.2f}%")
print(f"100-yr ≥1 m area vs 50-yr: {larger_vs_50:.1f}% larger")
print(f"100-yr ≥1 m area vs 20-yr: {larger_vs_20:.1f}% larger")


X (100-yr: % of inundated cells ≥1 m): 93.88%
Y (100-yr: % of entire domain ≥1 m): 93.80%
100-yr ≥1 m area vs 50-yr: 15.8% larger
100-yr ≥1 m area vs 20-yr: 98.8% larger


In [8]:
import numpy as np
import rasterio

# --- Input raster paths ---
r100 = r"D:\Phd Research\Final_Raster\100yr_compound_flood_base_stat.tif"
r50  = r"D:\Phd Research\Final_Raster\50-yr_compound_flood_stat.tif"
r20  = r"D:\Phd Research\Final_Raster\20yr_compound_flood_stat.tif"

# --- Depth threshold ---
DEPTH_THRESHOLD = 0.1 # meters

def inundation_area(path, threshold=1.0):
    with rasterio.open(path) as src:
        data = src.read(1).astype("float64")
        nodata = src.nodata
        valid = np.isfinite(data)
        if nodata is not None:
            valid &= (data != nodata)

        # Mask cells where depth >= threshold
        mask = valid & (data >= threshold)

        # Pixel area (in m²)
        pixel_area_m2 = abs(src.transform.a * src.transform.e)

        # Total inundation area (in m² and km²)
        inund_area_m2 = mask.sum() * pixel_area_m2
        inund_area_km2 = inund_area_m2 / 1e6

        return inund_area_km2

# --- Calculate for each file ---
area_100 = inundation_area(r100, DEPTH_THRESHOLD)
area_50  = inundation_area(r50, DEPTH_THRESHOLD)
area_20  = inundation_area(r20, DEPTH_THRESHOLD)

# --- Print results ---
print(f"Area (depth ≥ 0.1 m):")
print(f"  100-yr flood : {area_100:.2f} km²")
print(f"  50-yr flood  : {area_50:.2f} km²")
print(f"  20-yr flood  : {area_20:.2f} km²")



Area (depth ≥ 0.1 m):
  100-yr flood : 13959.96 km²
  50-yr flood  : 13379.88 km²
  20-yr flood  : 9657.68 km²


In [3]:
import rasterio
import numpy as np

# -------- USER INPUT --------
r_diff = r"D:\Phd Research\Final_Raster\diff_process_bath_100yr.tif"
TOL = 0.2     # ignore small differences within ±0.2 m
LIMIT = 12.0  # cap extreme outliers
# --------------------------------

# -------- READ AND CLEAN DATA --------
with rasterio.open(r_diff) as src:
    diff = src.read(1).astype(float)
    nodata = src.nodata if src.nodata is not None else -9999.0

mask = np.isfinite(diff) & (diff != nodata)
diff_valid = diff[mask]

# Clip to valid physical range
diff_valid = diff_valid[(diff_valid >= -LIMIT) & (diff_valid <= LIMIT)]

# Discard near-zero differences
diff_valid = diff_valid[(diff_valid <= -TOL) | (diff_valid >= TOL)]

# -------- AREA-BASED STATS --------
n_total = diff_valid.size
n_bathtub_higher = np.sum(diff_valid < -TOL)     # bathtub deeper
n_process_higher = np.sum(diff_valid > TOL)      # process deeper

pct_bathtub = 100 * n_bathtub_higher / n_total if n_total > 0 else np.nan
pct_process = 100 * n_process_higher / n_total if n_total > 0 else np.nan

print("=== Depth Difference Area Comparison ===")
print(f"Total valid cells used : {n_total:,}")
print(f"Cells where bathtub depth > process-based: {n_bathtub_higher:,} ({pct_bathtub:.2f}%)")
print(f"Cells where process-based depth > bathtub : {n_process_higher:,} ({pct_process:.2f}%)")
print(f"Ignored cells (|Δ| ≤ {TOL} m) are excluded from analysis.")


=== Depth Difference Area Comparison ===
Total valid cells used : 280,014
Cells where bathtub depth > process-based: 167,377 (59.77%)
Cells where process-based depth > bathtub : 112,637 (40.23%)
Ignored cells (|Δ| ≤ 0.2 m) are excluded from analysis.


In [7]:
import numpy as np
import rasterio

# ---------- USER INPUT ----------
cff_path = r"D:\Phd Research\Final_Raster\FAF_EPSG32614_V3.tif"
CLIP_TO_RANGE = True            # set False to skip clipping
CLIP_MIN, CLIP_MAX = -1.0, 1.0  # expected CFF/FAF range
# --------------------------------

with rasterio.open(cff_path) as src:
    cff = src.read(1).astype("float64")
    nd = src.nodata

    # Pixel area from affine (handles rotation/skew):
    # Use attributes (safe across rasterio versions)
    T = src.transform
    pix_area_m2 = abs(T.a * T.e - T.b * T.d)

# Mask nodata / non-finite
if nd is not None:
    cff = np.where(cff == nd, np.nan, cff)
valid = np.isfinite(cff)

# Optional clipping to plausible range
if CLIP_TO_RANGE:
    cff = np.where((cff < CLIP_MIN) | (cff > CLIP_MAX), np.nan, cff)
    valid = np.isfinite(cff)

# Compute stats
n_valid = int(np.count_nonzero(valid))
if n_valid == 0:
    raise RuntimeError("No valid CFF cells found after masking/clipping.")

weighted_sum = float(np.nansum(cff[valid] * pix_area_m2))   # Σ CFF_i * A_i
total_area_m2 = float(n_valid * pix_area_m2)
area_weighted_mean = weighted_sum / total_area_m2

# Pretty print
print("=== CFF Area-weighting ===")
print(f"Raster: {cff_path}")
print(f"Pixel area: {pix_area_m2:.6f} m²")
print(f"Valid cells: {n_valid:,}")
print(f"Total valid area: {total_area_m2/1e6:.3f} km²")
print(f"Weighted SUM (Σ CFF_i * A_i): {weighted_sum:.6f} m²")
print(f"Area-weighted MEAN CFF     : {area_weighted_mean:.6f} (dimensionless)")


=== CFF Area-weighting ===
Raster: D:\Phd Research\Final_Raster\FAF_EPSG32614_V3.tif
Pixel area: 40000.000000 m²
Valid cells: 347,285
Total valid area: 13891.400 km²
Weighted SUM (Σ CFF_i * A_i): -1087268137.432846 m²
Area-weighted MEAN CFF     : -0.078269 (dimensionless)
